In [55]:
# Imports
import pandas as pd
import numpy as np
import spacy
from dateparser.search import search_dates
from dateutil.parser import ParserError
from dateutil import parser as dateutil_parser
from datetime import datetime
import re


# Loading the csv into a dataframe and selecting the columns of interest
file_path = r'_github-AAC_accidents_tagged_data.xlsx'
df_long = pd.read_excel(file_path)
df = df_long[['ID', 'Accident Title', 'Publication Year', 'Text', 'Tags Applied']].copy()


In [62]:
# extracting location tags

test_text = df.iat[4, 3]
test_loc = df.iat[4,1]

# United States
US_STATES_FULL = [
    "Alabama","Alaska","Arizona","Arkansas","California","Colorado","Connecticut",
    "Delaware","Florida","Georgia","Hawaii","Idaho","Illinois","Indiana","Iowa",
    "Kansas","Kentucky","Louisiana","Maine","Maryland","Massachusetts","Michigan",
    "Minnesota","Mississippi","Missouri","Montana","Nebraska","Nevada",
    "New Hampshire","New Jersey","New Mexico","New York","North Carolina",
    "North Dakota","Ohio","Oklahoma","Oregon","Pennsylvania","Rhode Island",
    "South Carolina","South Dakota","Tennessee","Texas","Utah","Vermont",
    "Virginia","Washington","West Virginia","Wisconsin","Wyoming",
    "District of Columbia"
]

# Canada
CAN_PROVINCES_FULL = [
    "Alberta","British Columbia","Manitoba","New Brunswick",
    "Newfoundland and Labrador","Nova Scotia","Ontario",
    "Prince Edward Island","Quebec","Saskatchewan",
    "Northwest Territories","Nunavut","Yukon", "Baffin Island"
]

# Mexico
MEX_STATES_FULL = [
    "Aguascalientes","Baja California","Baja California Sur","Campeche","Chiapas",
    "Chihuahua","Coahuila","Colima","Durango","Guanajuato","Guerrero","Hidalgo",
    "Jalisco","Mexico City","México","Michoacán","Morelos","Nayarit",
    "Nuevo León","Oaxaca","Puebla","Querétaro","Quintana Roo","San Luis Potosí",
    "Sinaloa","Sonora","Tabasco","Tamaulipas","Tlaxcala","Veracruz",
    "Yucatán","Zacatecas"
]


# Lookup Dictionary
REGION_LOOKUP = {}

for state in US_STATES_FULL:
    REGION_LOOKUP[state.lower()] = ("US", state)

for prov in CAN_PROVINCES_FULL:
    REGION_LOOKUP[prov.lower()] = ("CA", prov)

for state in MEX_STATES_FULL:
    REGION_LOOKUP[state.lower()] = ("MX", state)

# Build regex pattern (longer names first to prevent partial matching issues)
sorted_regions = sorted(REGION_LOOKUP.keys(), key=lambda x: -len(x))
pattern = r'\b(?:' + '|'.join(re.escape(name) for name in sorted_regions) + r')\b'
REGION_REGEX = re.compile(pattern, flags=re.IGNORECASE)


# Extraction Function
def extract_state_province(text: str):
    if not isinstance(text, str) or not text.strip():
        return None

    match = REGION_REGEX.search(text)

    if match:
        region = match.group(0).lower()
        return REGION_LOOKUP[region]

    return None


# Add the locations to our dataframe
df['Country'] = df['Accident Title'].apply(
    lambda x: extract_state_province(x)[0] 
    if extract_state_province(x) is not None 
    else None
)

df['State_Province'] = df['Accident Title'].apply(
    lambda x: extract_state_province(x)[1] 
    if extract_state_province(x) is not None 
    else None
)

# If the location was not in the title, try again in the text
mask = df['Country'].isna()
df.loc[mask, 'Country'] = df.loc[mask, 'Text'].apply(lambda x: extract_state_province(x)[0]if extract_state_province(x) is not None else np.nan)
mask = df['State_Province'].isna()
df.loc[mask, 'State_Province'] = df.loc[mask, 'Text'].apply(lambda x: extract_state_province(x)[1]if extract_state_province(x) is not None else np.nan)

In [66]:
df.to_csv('ANAC_loc.csv', index=False)

